# Clean-Slate — a quantitative teardown 🔬
### Causal residualisation · residual-WML CAPM alpha (HAC) · null-seed battery · paired residual-vs-total bootstrap · the defence stack

![Signal: Weak](https://img.shields.io/badge/Signal-Weak-dab617?style=flat-square)
![Tradability: Fragile](https://img.shields.io/badge/Tradability-Fragile-dab617?style=flat-square)
![Cleaner than total?: Unproven here](https://img.shields.io/badge/Cleaner_than_total%3F-Unproven_here-8b949e?style=flat-square)

The deep companion to the [notebook for the curious](01_for_the_curious.ipynb) — *same seven beats, every claim with its standard error.* The steelman is §3.7 residual momentum (Blitz, Huij & Martens 2011): 12-1 momentum on factor-residual returns. We use a 1-factor (market) residual — a stated simplification — prove the engine on a synthetic panel, then compare to total momentum on the real S&P 500, **as a paired test of the differences**, not a beauty contest of two insignificant point estimates.

> ⚠️ **Not investment advice.** The core executes on synthetic data; the real run is in [`../docs/results.md`](../docs/results.md), sources in [`../docs/references.md`](../docs/references.md).
>
> 💡 **The `💡 In plain words` notes** translate each result back to intuition.

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))
sys.path.insert(0, os.path.abspath("../../.."))
%matplotlib inline
import matplotlib.pyplot as plt
plt.rcParams["figure.figsize"] = (9.5, 5.2)
import numpy as np, pandas as pd
pd.set_option("display.float_format", lambda v: f"{v:,.4f}")
from clean_slate import data, momentum, strategy, decompose, extension

# Offline synthetic panel: a MOMENTUM tape (persistent idiosyncratic drift -> the *residual* carries the
# momentum) and a no-momentum NULL. The real S&P 500 verdict is in ../docs/results.md.
panel,  market,  truth = data.synthetic_panel(mom_strength=0.0016, seed=25)   # the momentum tape
panel0, market0, _     = data.synthetic_panel(mom_strength=0.0,    seed=25)   # the null
print(f"{truth.n_stocks} stocks x {truth.n_bars} days | baked mom_strength={truth.mom_strength} | null=0")


150 stocks x 4032 days | baked mom_strength=0.0016 | null=0


## Beat 0 · Verdict

| Axis | Stamp | Why |
|---|---|---|
| **Signal** — is residual momentum real? | 🟡 `WEAK` | Strong on the control (alpha HAC *t* ≈ 16); on the modern S&P 500 the residual-WML alpha is **+5.0%/yr** (*t* = **+1.0**) — a higher point estimate than total momentum (+4.4%/t+0.9) but still insignificant. |
| **Tradability** | 🟡 `FRAGILE` | Standalone Sharpe (**-0.01**), fast turnover (**12×/yr**), short-the-losers. |
| **Cleaner than total?** | ⚪ `Unproven here` | The paired block-bootstrap of the gaps on the books' common 174 months: skew gap **-0.19** (95% CI **[-0.88, +0.44]**), Sharpe gap **-0.12** (95% CI **[-0.28, +0.02]**) — neither clears zero, and both lean the *wrong* way. |

> **In one sentence:** residualising momentum raises the alpha point estimate and gives a clean platform for crash management — but under a paired test the "cleaner cousin" advantage doesn't clear zero on this tape, and what reliably tames the crash (**-67% → -30%**) is the vol-management stacked on top.

*(This notebook executes on synthetic panels; the real S&P numbers are in [`../docs/results.md`](../docs/results.md).)*

## Beat 1 · The claim, precisely

Residualise with a trailing (causal) regression: $\hat\varepsilon_{i,t} = r_{i,t} - \hat\beta_{i,t-1}\,r_{\text{mkt},t}$, $\hat\beta$ a rolling slope. Score $m_i = \prod_{t-252}^{t-21}(1+\hat\varepsilon_i)-1$; WML = top-minus-bottom residual decile. The synthetic bakes the persistent drift into $\theta$ (the residual); the market carries beta dispersion. The source uses FF3 residuals — we use 1-factor (market).

In [2]:
rr = momentum.residual_returns(panel, market)
print(f"residual market-correlation {rr.mean(axis=1).corr(market):+.2f} "
      f"vs raw {panel.mean(axis=1).corr(market):+.2f} -- the residual strips most of the market")

residual market-correlation +0.00 vs raw +1.00 -- the residual strips most of the market


## Beat 2 · So what?

Momentum's crash is concentrated in its *systematic* exposure: after a market crash, the winners are low-beta defensives and the losers high-beta cyclicals, so a momentum book ends up implicitly short the market right before it rebounds (Daniel–Moskowitz 2016). Residualising removes exactly that bet. The open questions: how much premium survives, whether the residual book is *measurably* cleaner than the total book on the same window, and whether stacking vol-management finishes the job. Beats 4–6 answer all three.

## Beat 3 · Pre-registered protocol

1. **Residual premium** (`decompose.capm_alpha`): residual-WML alpha + HAC *t*.
2. **Cleaner?** (`decompose.crash_comparison` + `decompose.paired_crash_bootstrap`): the side-by-side profiles, then a **paired** circular block-bootstrap of the skew and Sharpe gaps on the books' common months — the third-axis stamp rests on the paired test, never on two standalone point estimates.
3. **Stack** (`extension.defence_stack`): total → residual → residual + vol-managed.
4. **Null calibration** (`decompose.null_alpha_battery`): no persistence ⇒ no premium, asserted on a *battery* of no-momentum seeds (one tape is one draw).

**Confirmed line:** the residual premium is significant, *and* a paired gap clears zero in the residual's favour, *and* the stack tames the drawdown.

## Beat 4 · The teardown

### 4a · Residual alpha, momentum vs null

In [3]:
for label, (p, mk) in [('momentum', (panel, market)), ('null', (panel0, market0))]:
    a = decompose.capm_alpha(p, mk, cost_bps=5.0)
    print(f"{label:9s}: residual alpha {a['alpha_ann_pct']:+.1f}%/yr (HAC t {a['alpha_t']:+.1f}), "
          f"beta {a['beta']:+.2f}, Sharpe {a['sharpe']:+.2f}, skew {a['skew']:+.2f}")

momentum : residual alpha +32.2%/yr (HAC t +15.8), beta -0.01, Sharpe +4.43, skew +0.01


null     : residual alpha -4.2%/yr (HAC t -2.1), beta +0.02, Sharpe -0.53, skew -0.06


> 💡 **In plain words.** The momentum tape's alpha (*t* ≈ 16) is the machine working. The null tape's *negative* alpha (≈ −4%/yr, *t* ≈ −2) is **not** a broken harness: a single 16-year no-momentum tape is one draw (per-seed alpha sd ≈ 1.6%/yr, *t* sd ≈ 0.9), and the net book pays a ~0.7%/yr cost drag at 12×/yr turnover. Calibrate, don't eyeball — the battery below includes both observed tail seeds (24 and 25) and must centre on ≈ 0 gross:

In [4]:
bat = decompose.null_alpha_battery(seeds=(0, 1, 2, 24, 25), cost_bps=5.0)
display(bat.round(2))
print(f"null battery: mean alpha {bat['alpha_ann_pct'].mean():+.2f}%/yr (mean HAC t {bat['alpha_t'].mean():+.2f}) "
      f"-- centred on ~0 net of the cost drag; max |t| {bat['alpha_t'].abs().max():.2f}")

,alpha_ann_pct,alpha_t
seed,,
0,-1.3900,-0.7300
1,0.6700,0.3600
2,-0.3200,-0.1700
24,3.4700,1.8000
25,-4.1600,-2.1300


null battery: mean alpha -0.35%/yr (mean HAC t -0.17) -- centred on ~0 net of the cost drag; max |t| 2.13


### 4b · Residual vs total — the crash comparison, then the paired test

In [5]:
cc = decompose.crash_comparison(panel, market, cost_bps=5.0)
display(pd.DataFrame(cc).T.round(2))
print('On the REAL S&P 500: residual skew -0.23 (vs total -0.04), drawdown -70% (vs -67%).')

,sharpe,skew,worst_month_pct,max_drawdown_pct
residual,4.4300,0.2500,-2.8000,-4.6500
total,4.0700,0.0700,-3.8500,-7.8200


On the REAL S&P 500: residual skew -0.23 (vs total -0.04), drawdown -70% (vs -67%).


Side-by-side profiles invite eyeballing — but "cleaner" is a claim about a **difference** between two books that trade the same names on the same days, so it gets a **paired** circular block-bootstrap (6-month blocks — momentum-book months are autocorrelated) of the aligned monthly pairs. On the baked tape the residual book *must* win, and the test must see it:

In [6]:
pb = decompose.paired_crash_bootstrap(panel, market, n_boot=2000, seed=0, cost_bps=5.0)
print(f"paired gaps (residual - total), {pb['n_months']} months, {pb['n_boot']} resamples, {pb['block_months']}-mo blocks:")
print(f"  Sharpe gap {pb['sharpe_diff']:+.2f}  95% CI [{pb['sharpe_ci_low']:+.2f}, {pb['sharpe_ci_high']:+.2f}]  "
      f"P(residual better) {pb['sharpe_frac_residual_better']:.0%}  significant: {pb['sharpe_significant']}")
print(f"  skew   gap {pb['skew_diff']:+.2f}  95% CI [{pb['skew_ci_low']:+.2f}, {pb['skew_ci_high']:+.2f}]  "
      f"P(residual better) {pb['skew_frac_residual_better']:.0%}  significant: {pb['skew_significant']}")
print('On the REAL S&P 500 (174 months): skew gap -0.19 [-0.88, +0.44] (P 36%), '
      'Sharpe gap -0.12 [-0.28, +0.02] (P 5%) -- neither clears zero.')

paired gaps (residual - total), 163 months, 2000 resamples, 6-mo blocks:
  Sharpe gap +0.51  95% CI [+0.04, +0.92]  P(residual better) 98%  significant: True
  skew   gap +0.18  95% CI [-0.14, +0.52]  P(residual better) 87%  significant: False
On the REAL S&P 500 (174 months): skew gap -0.19 [-0.88, +0.44] (P 36%), Sharpe gap -0.12 [-0.28, +0.02] (P 5%) -- neither clears zero.


> 💡 **In plain words.** On the synthetic tape — where the momentum is planted in the residual by construction — the paired Sharpe gap clears zero (the residual book wins in ~98% of resamples); the skew gap is directional only. On the **real** S&P 500 *nothing* clears zero, and both gaps lean the wrong way (skew -0.19, Sharpe -0.12). The higher residual *alpha* there reflects the book's negative market beta (-0.29) in a rising market as much as any extra momentum content. That is why the third axis reads `Unproven here`, not `Confirmed`.

### 4c · The defence stack

In [7]:
st = extension.defence_stack(panel, market, cost_bps=5.0)
display(pd.DataFrame(st).T.round(2))
print('On the REAL S&P 500: drawdown -67% (total) -> -70% (residual alone) -> -30% (stacked, Sharpe +0.16).')

,sharpe,skew,worst_month_pct,max_drawdown_pct
total,4.0700,0.0700,-3.8500,-7.8200
residual,4.4300,0.2500,-2.8000,-4.6500
residual_vol_managed,4.3600,0.3600,-5.1400,-7.0100


On the REAL S&P 500: drawdown -67% (total) -> -70% (residual alone) -> -30% (stacked, Sharpe +0.16).


## Beat 5 · The verdict

- **Real on control** (4a): residual alpha HAC *t* ≈ 16; the null battery is centred on ≈ 0.
- **Faint here** (4b): alpha +5.0%/yr (*t* +1.0); the paired gaps (skew -0.19, Sharpe -0.12) straddle zero and lean the wrong way.
- **Stack tames the tail** (4c): drawdown -67% → -30% at Sharpe +0.16.

> **Signal `WEAK` · Tradability `FRAGILE` · Cleaner than total momentum? `Unproven here`.**

## Beat 6 · Could you trade it?

- **The standalone book earns nothing** (Sharpe -0.01); short-the-losers; 12×/yr turnover.
- **The right platform for crash management** — stacking vol-targeting drops the drawdown to **-30%** — though [Study 24](../../24-stampede/) shows vol-management alone gets most of the way on the total book (−61% → −32% on its full window).
- **A 1-factor residual under-cleans** — FF3 residuals would shed more of the value-driven crash.

Tradability **`FRAGILE`**; cleaner than total `Unproven here`.

## Beat 7 · Going further

### 7a · Worked complement — the defence stack
Total-WML vs residual-WML vs vol-managed residual-WML: where the tail actually yields.

In [8]:
st = extension.defence_stack(panel, market, cost_bps=5.0)
for k in ['total', 'residual', 'residual_vol_managed']:
    p = st[k]; print(f"{k:22s}: Sharpe {p['sharpe']:+.2f}, skew {p['skew']:+.2f}, "
                     f"worst month {p['worst_month_pct']:+.1f}%, max drawdown {p['max_drawdown_pct']:.0f}%")
print('On the REAL S&P 500 (../docs/extension.md): drawdown -67% -> -70% -> -30%.')

total                 : Sharpe +4.07, skew +0.07, worst month -3.8%, max drawdown -8%
residual              : Sharpe +4.43, skew +0.25, worst month -2.8%, max drawdown -5%
residual_vol_managed  : Sharpe +4.36, skew +0.36, worst month -5.1%, max drawdown -7%
On the REAL S&P 500 (../docs/extension.md): drawdown -67% -> -70% -> -30%.


**The result.** Residualising alone does not dent the real-tape drawdown (**-70%** vs total **-67%** — the value-driven part of the crash needs factors we don't have); vol-scaling removes the regime-clustered tail; and **stacked** the drawdown falls from **-67%** to **-30%** while the Sharpe *rises* to **+0.16**. The crash, the thing that made [Study 24](../../24-stampede/) `FRAGILE`, is engineerable — but that is the overlay's work as much as the residual's, and the faint *premium* on the modern large-cap sample is the part that isn't. Full run in [`../docs/extension.md`](../docs/extension.md).

### 7b · Other forks
- **FF3 residuals** (MKT, SMB, HML) — the source's recipe; should shed the value-driven crash the 1-factor residual leaves behind. Rerun `paired_crash_bootstrap` on them.
- **Industry-/sector-neutral momentum** — an alternative neutralisation; compare the crash.
- **Constant-volatility momentum** (Barroso–Santa-Clara) applied to the residual factor.

PRs welcome — add the FF3 residual, or benchmark the neutralisations head-to-head.